In [0]:
from datetime import datetime

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','parquet')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option('cloudFiles.schemaLocation','/Volumes/projectdev/bronze/unique_carriers/schema')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/UNIQUE_CARRIERS/{datetime.now().strftime('year=%Y/month=%m/day=%d')}/')

In [0]:
display(
    df,
    checkpointLocation="/Volumes/projectdev/bronze/unique_carriers/checkpoint1/"
)

In [0]:
dbutils.fs.rm('/Volumes/projectdev/bronze/unique_carriers/checkpoint/', True)

In [0]:
df_clean = df.selectExpr("replace(Code, '\"', '') as Code",
                         "replace(Description, '\"', '') as Description"
                         )

df_clean.writeStream.trigger(once=True)\
    .format('delta')\
    .option("mergeSchema", "true")\
    .outputMode('append')\
    .option('checkpointLocation','/Volumes/projectdev/bronze/unique_carriers/checkpoint/')\
    .start('abfss://cleansed@datalakestrgdev.dfs.core.windows.net/unique_carriers')
     

In [0]:
%sql
create table if not exists projectdev.cleansed.unique_carriers
using delta
location 'abfss://cleansed@datalakestrgdev.dfs.core.windows.net/unique_carriers'

In [0]:
%sql
select * from projectdev.cleansed.unique_carriers